# 3DBreastNet - 128x128x128 Voxel Reconstruction

## 1. Install Dependencies

In [ ]:
!uv pip install torch torchvision tifffile opencv-python numpy scipy scikit-image matplotlib tqdm pandas


In [ ]:
# tqdm.notebook is provided by tqdm; ipywidgets enables the notebook progress UI.
!uv pip install tqdm ipywidgets


## 2. Model Definitions

In [1]:
# ════════════════════════════════════════════════════════════════════════════
# ## CELL A — New U-Net Model with Skip Connections
# ════════════════════════════════════════════════════════════════════════════
# Paste this as a NEW CODE CELL right after the existing model definitions cell.

"""BreastNet3D_UNet — Encoder-Decoder with 2D→3D skip connections."""
import math, torch, torch.nn as nn, torch.nn.functional as F

# ── Skip Projection: bridges 2D encoder features to 3D decoder space ──
class SkipProjection(nn.Module):
    """Projects a 2D feature map (B,C,H,W) to 3D (B,out_ch,D,H,W)."""
    def __init__(self, in_ch, out_ch, depth):
        super().__init__()
        self.depth = depth
        self.proj = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, kernel_size=1, bias=False),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, feat_2d):
        feat_3d = feat_2d.unsqueeze(2).expand(-1, -1, self.depth, -1, -1)
        return self.proj(feat_3d)

# ── Decoder block with skip concatenation ──
class DecoderBlock3D(nn.Module):
    """Upsample + concat skip + double conv."""
    def __init__(self, in_ch, skip_ch, out_ch, drop=0.0):
        super().__init__()
        self.up = nn.ConvTranspose3d(in_ch, in_ch, kernel_size=2, stride=2)
        self.block = nn.Sequential(
            nn.Conv3d(in_ch + skip_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm3d(out_ch), nn.ReLU(True),
            nn.Dropout3d(drop) if drop > 0 else nn.Identity(),
            nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm3d(out_ch), nn.ReLU(True),
        )
    def forward(self, x, skip=None):
        x = self.up(x)
        if skip is not None:
            if x.shape[2:] != skip.shape[2:]:
                x = F.interpolate(x, size=skip.shape[2:],
                                  mode='trilinear', align_corners=False)
            x = torch.cat([x, skip], dim=1)
        return self.block(x)

# ── Combined U-Net Model ──
# Architecture: 6-stage 2D encoder → 1000-d bottleneck → 6-stage 3D decoder
# Skip connections on enc2–enc5 (4 levels). enc1 skipped (128³ too large),
# enc6 skipped (too close to bottleneck).
#
# Skip channel budget (memory-safe for 128³):
#   enc5 (512ch, 8²)  → proj5 → 32ch @ 8³   → cat with dec stage 2
#   enc4 (256ch, 16²) → proj4 → 16ch @ 16³  → cat with dec stage 3
#   enc3 (128ch, 32²) → proj3 → 8ch  @ 32³  → cat with dec stage 4
#   enc2 (64ch,  64²) → proj2 → 4ch  @ 64³  → cat with dec stage 5

class BreastNet3D_UNet(nn.Module):
    def __init__(self, drop=0.25):
        super().__init__()
        # ── 2D Encoder (same weights as Encoder2D) ──
        self.pool = nn.MaxPool2d(2)
        self.enc1 = DoubleConv2D(5, 32, 0)
        self.enc2 = DoubleConv2D(32, 64, 0)
        self.enc3 = DoubleConv2D(64, 128, drop)
        self.enc4 = DoubleConv2D(128, 256, drop)
        self.enc5 = DoubleConv2D(256, 512, drop)
        self.enc6 = DoubleConv2D(512, 512, drop)
        self.enc_fc = nn.Sequential(nn.Dropout(drop), nn.Linear(512*2*2, 1000))

        # ── Skip Projections (2D → 3D) ──
        self.proj2 = SkipProjection(64,  4,  depth=64)
        self.proj3 = SkipProjection(128, 8,  depth=32)
        self.proj4 = SkipProjection(256, 16, depth=16)
        self.proj5 = SkipProjection(512, 32, depth=8)

        # ── Bottleneck ──
        self.dec_fc = nn.Linear(1000, 512*2*2*2)

        # ── 3D Decoder with skip injection ──
        # Stage 1: 2³→4³, no skip
        self.up1 = nn.ConvTranspose3d(512, 256, 2, stride=2)
        self.d1  = DoubleConv3D(256, 256, drop)
        # Stage 2: 4³→8³, skip from enc5 (32ch)
        self.dec2 = DecoderBlock3D(256, 32, 128, drop)
        # Stage 3: 8³→16³, skip from enc4 (16ch)
        self.dec3 = DecoderBlock3D(128, 16, 64, drop)
        # Stage 4: 16³→32³, skip from enc3 (8ch)
        self.dec4 = DecoderBlock3D(64, 8, 32, 0)
        # Stage 5: 32³→64³, skip from enc2 (4ch)
        self.dec5 = DecoderBlock3D(32, 4, 16, 0)
        # Stage 6: 64³→128³, no skip
        self.up6 = nn.ConvTranspose3d(16, 8, 2, stride=2)
        self.d6  = DoubleConv3D(8, 8, 0)

        # ── Visual hull fusion (identical to Decoder3D) ──
        self.fusion = nn.Sequential(
            nn.Conv3d(8 + 1, 8, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm3d(8), nn.ReLU(True),
            nn.Conv3d(8, 1, kernel_size=1),
            nn.Sigmoid()
        )
        self.apply(_init)
        if self.fusion[3].bias is not None:
            nn.init.constant_(self.fusion[3].bias, -4.0)

    def forward(self, x, visual_hull=None):
        # ── Encoder (store skip features before pool) ──
        s1 = self.enc1(x)              # (B, 32,  128, 128)
        s2 = self.enc2(self.pool(s1))  # (B, 64,   64,  64)
        s3 = self.enc3(self.pool(s2))  # (B, 128,  32,  32)
        s4 = self.enc4(self.pool(s3))  # (B, 256,  16,  16)
        s5 = self.enc5(self.pool(s4))  # (B, 512,   8,   8)
        s6 = self.enc6(self.pool(s5))  # (B, 512,   4,   4)
        z  = self.enc_fc(self.pool(s6).view(x.size(0), -1))  # (B, 1000)

        # ── Bottleneck → 3D seed ──
        x3 = self.dec_fc(z).view(x.size(0), 512, 2, 2, 2)

        # ── Decoder with skip connections ──
        x3 = self.d1(self.up1(x3))                    # (B,256, 4³)
        x3 = self.dec2(x3, self.proj5(s5))             # (B,128, 8³)
        x3 = self.dec3(x3, self.proj4(s4))             # (B, 64,16³)
        x3 = self.dec4(x3, self.proj3(s3))             # (B, 32,32³)
        x3 = self.dec5(x3, self.proj2(s2))             # (B, 16,64³)
        x3 = self.d6(self.up6(x3))                     # (B,  8,128³)

        # ── Visual hull fusion ──
        if visual_hull is None:
            visual_hull = torch.zeros(
                x3.size(0), 1, x3.size(2), x3.size(3), x3.size(4),
                device=x3.device, dtype=x3.dtype)
        x3 = torch.cat([x3, visual_hull], dim=1)
        return self.fusion(x3) * visual_hull

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# ## CELL B — Shape Prior (defined but NOT wired into training)
# ════════════════════════════════════════════════════════════════════════════

def build_dual_ellipsoid_prior(volume_size=128, device='cuda'):
    V = volume_size
    zz, yy, xx = torch.meshgrid(
        torch.linspace(0, 1, V, device=device),
        torch.linspace(0, 1, V, device=device),
        torch.linspace(0, 1, V, device=device),
        indexing='ij')
    cx_L, cx_R = 0.30, 0.70
    cy, cz = 0.55, 0.50
    ax, ay, az = 0.20, 0.35, 0.28
    def ellipsoid_sdf(cx):
        return ((xx-cx)**2/ax**2 + (yy-cy)**2/ay**2 + (zz-cz)**2/az**2)
    e_L = ellipsoid_sdf(cx_L)
    e_R = ellipsoid_sdf(cx_R)
    prior = torch.sigmoid(8.0 * (1.0 - torch.minimum(e_L, e_R)))
    return prior.unsqueeze(0).unsqueeze(0)  # (1,1,V,V,V)

class ShapePriorLoss(nn.Module):
    def __init__(self, volume_size=128, device='cuda', weight=0.3):
        super().__init__()
        self.weight = weight
        prior = build_dual_ellipsoid_prior(volume_size, device).detach()
        self.register_buffer('prior', prior)
    def forward(self, V_pred):
        outside_mass = V_pred * (1.0 - self.prior)
        return self.weight * outside_mass.mean()

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# ## CELL C — Runtime Flags
# ════════════════════════════════════════════════════════════════════════════

USE_SHAPE_PRIOR = False  # set True to activate Upgrade 2

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# ## CELL D — Verification: U-Net Forward Pass
# ════════════════════════════════════════════════════════════════════════════

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_unet = BreastNet3D_UNet().to(device)
dummy = torch.zeros(2, 5, 128, 128, device=device)
# Need a dummy visual hull too
dummy_hull = torch.ones(2, 1, 128, 128, 128, device=device)
with torch.no_grad():
    out = model_unet(dummy, dummy_hull)
assert out.shape == (2, 1, 128, 128, 128), f"Shape mismatch: {out.shape}"
print("U-Net model forward pass OK:", out.shape)
del model_unet, dummy, dummy_hull
torch.cuda.empty_cache()

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# ## CELL E — Verification: Prior Visualisation
# ════════════════════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt
prior_np = build_dual_ellipsoid_prior(128, 'cpu').squeeze().numpy()
plt.imshow(prior_np[64, :, :], cmap='hot', vmin=0, vmax=1)
plt.title('Dual-ellipsoid prior — frontal slice (depth=64)')
plt.colorbar()
plt.show()

## 3. Training

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# ## CELL F — Modified Training Loop (REPLACES existing training cell)
# ════════════════════════════════════════════════════════════════════════════
# Key changes vs original:
#   - Uses BreastNet3D_UNet (single model) instead of separate enc/dec
#   - Saves to 3dbreastnet_unet_best.pth / 3dbreastnet_unet_last.pth
#   - History keys: train_dice_loss, val_dice_loss, val_dice_score, val_hd95, prior_loss
#   - Conditional shape prior integration via USE_SHAPE_PRIOR flag
# ════════════════════════════════════════════════════════════════════════════

"""3DBreastNet — Training script for 128³ voxel reconstruction (U-Net edition)."""
import os, sys, time, random, json, math
import numpy as np, cv2, tifffile, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict
from tqdm.auto import tqdm

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import scipy.ndimage

# ════════════════════════════════════════════════════════════════
# CONFIG
# ════════════════════════════════════════════════════════════════
CFG = {
    "epochs":       400,
    "batch_size":   2,
    "lr":           1e-4,
    "betas":        (0.5, 0.9),
    "n_per_view":   2,
    "seed":         42,
    "patience":     50,
    "ckpt_dir":     "checkpoints_3d_v6",
    "tiff_base":    r"../../data/organized_by_patient",
    "unet_ckpt":    r"/mnt/Data1/Peoples/faiz836b/DNP-3DDMR-IR/UNET_Segmentation/breast_segmentation_unet_best_gpu.pth",
}

# ════════════════════════════════════════════════════════════════
# DATA (unchanged from v6)
# ════════════════════════════════════════════════════════════════
@dataclass
class PatientGroup:
    patient_id: str
    label: str
    views: Dict[str, Path]

def get_view_key(filename):
    n = filename.lower()
    if "right later" in n: return "RL"
    if "right obli"  in n: return "RO"
    if "frontal" in n or "anterior" in n: return "F"
    if "left obliq"  in n: return "LO"
    if "left later"  in n: return "LL"
    return None

def build_patient_groups(tiff_base):
    tb = Path(tiff_base)
    pd_ = {}
    for tp in tb.rglob("*.tiff"):
        parts = tp.relative_to(tb).parts
        if len(parts) < 3: continue
        pid, lab, fn = parts[0], parts[1], parts[-1]
        vk = get_view_key(fn)
        if not vk: continue
        key = (pid, lab)
        if key not in pd_: pd_[key] = {"views": {}}
        pd_[key]["views"][vk] = tp
    groups, skip, nb_, nm = [], 0, 0, 0
    for (pid, lab), d in pd_.items():
        if len(d["views"]) == 5:
            groups.append(PatientGroup(pid, lab, d["views"]))
            nb_ += lab.lower() == "benign"; nm += lab.lower() != "benign"
        else:
            print(f"  Skip {pid} ({lab}): {len(d['views'])}/5 views"); skip += 1
    groups.sort(key=lambda g: g.patient_id)
    print(f"Patients: {len(pd_)} | Complete: {len(groups)} | "
          f"Skipped: {skip} | B={nb_} M={nm}")
    return groups

class PatientDataset(Dataset):
    def __init__(self, groups, unet, device, img_sz=256):
        self.groups, self.unet, self.device = groups, unet, device
        self.img_sz = img_sz
        self.views = ["RL","RO","F","LO","LL"]
    def __len__(self): return len(self.groups)
    def __getitem__(self, idx):
        g = self.groups[idx]; thermals, masks = [], []
        for v in self.views:
            raw = tifffile.imread(str(g.views[v])).astype(np.float32)
            raw = cv2.resize(raw, (self.img_sz, self.img_sz))
            mn, mx = raw.min(), raw.max()
            norm = (raw - mn) / (mx - mn + 1e-8)
            thermals.append(norm)
            with torch.no_grad():
                inp = torch.tensor(norm).unsqueeze(0).unsqueeze(0).to(self.device)
                m = (torch.sigmoid(self.unet(inp)).squeeze().cpu().numpy() > 0.5).astype(np.float32)
            masks.append(cv2.resize(m, (128,128), interpolation=cv2.INTER_NEAREST))
        return {
            "masks_5ch": torch.tensor(np.stack(masks), dtype=torch.float32),
            "thermals_5ch": torch.tensor(np.stack(thermals), dtype=torch.float32),
            "patient_id": g.patient_id, "label": g.label,
        }

# ════════════════════════════════════════════════════════════════
# METRICS (unchanged)
# ════════════════════════════════════════════════════════════════
def hd95(p, t):
    if p.sum()==0 or t.sum()==0: return 128.0
    pe = p ^ scipy.ndimage.binary_erosion(p)
    te = t ^ scipy.ndimage.binary_erosion(t)
    dtp = scipy.ndimage.distance_transform_edt(~pe)
    dtt = scipy.ndimage.distance_transform_edt(~te)
    d1 = np.percentile(dtt[pe], 95) if pe.sum()>0 else 128.0
    d2 = np.percentile(dtp[te], 95) if te.sum()>0 else 128.0
    return max(d1, d2)

# ════════════════════════════════════════════════════════════════
# TRAIN (modified for BreastNet3D_UNet)
# ════════════════════════════════════════════════════════════════
def train(cfg):
    torch.manual_seed(cfg["seed"]); np.random.seed(cfg["seed"])
    torch.cuda.manual_seed_all(cfg["seed"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    # U-Net (frozen segmentor)
    unet = UNet().to(device)
    unet.load_state_dict(torch.load(cfg["unet_ckpt"], map_location=device))
    unet.eval()
    for p in unet.parameters(): p.requires_grad = False

    # Data
    groups = build_patient_groups(cfg["tiff_base"])
    rng = random.Random(cfg["seed"])
    ben = [g for g in groups if g.label.lower()=="benign"]
    mal = [g for g in groups if g.label.lower()!="benign"]
    rng.shuffle(ben); rng.shuffle(mal)
    s = 0.78
    trn = ben[:int(len(ben)*s)] + mal[:int(len(mal)*s)]
    val = ben[int(len(ben)*s):] + mal[int(len(mal)*s):]
    print(f"Train: {len(trn)} | Val: {len(val)}")

    trn_dl = DataLoader(PatientDataset(trn, unet, device),
                        batch_size=cfg["batch_size"], shuffle=True, drop_last=True)
    val_dl = DataLoader(PatientDataset(val, unet, device),
                        batch_size=cfg["batch_size"], shuffle=False)

    # ── Model: BreastNet3D_UNet (single combined model) ──
    model = BreastNet3D_UNet().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"], betas=cfg["betas"])
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode="max", factor=0.5, patience=30, min_lr=1e-6)
    scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())
    Path(cfg["ckpt_dir"]).mkdir(parents=True, exist_ok=True)

    # ── Optional shape prior ──
    shape_prior_loss = None
    if USE_SHAPE_PRIOR:
        shape_prior_loss = ShapePriorLoss(volume_size=128, device=device)
        print("Shape prior ENABLED (weight={})".format(shape_prior_loss.weight))

    best_dice, no_imp = 0.0, 0
    hist = {"epoch":[], "train_dice_loss":[], "val_dice_loss":[],
            "val_dice_score":[], "val_hd95":[], "prior_loss":[]}
    val_angles = [-90., -45., 0., 45., 90.]

    for epoch in range(1, cfg["epochs"]+1):
        t0 = time.time()
        # ── train ──
        model.train()
        ep_loss, ep_prior = 0.0, 0.0
        for batch in tqdm(trn_dl, desc=f"E{epoch:03d} train", leave=False):
            m5 = batch["masks_5ch"].to(device)
            B = m5.size(0)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
                hull = compute_visual_hull(m5, device)
                vol = model(m5, hull)

            vol = vol.float()
            loss = torch.tensor(0.0, device=device)
            for i in range(5):
                lo, hi = VIEW_WINDOWS[i]
                for _ in range(cfg["n_per_view"]):
                    th = torch.rand(B, device=device)*(hi-lo)+lo
                    proj = render_projection(vol, th)
                    gt_mask = m5[:, i:i+1]
                    dl = dice_loss(proj, gt_mask)
                    bl = boundary_loss(proj, gt_mask)
                    loss = loss + dl + (2.0 * bl)

            loss = loss / (5*cfg["n_per_view"])

            # Conditional shape prior
            prior_val = 0.0
            if USE_SHAPE_PRIOR and shape_prior_loss is not None:
                pl = shape_prior_loss(vol)
                loss = loss + pl
                prior_val = pl.item()

            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            if torch.isfinite(loss):
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt)
            else:
                print(f"  ⚠ NaN loss in epoch {epoch}, skipping batch")
            scaler.update()
            opt.zero_grad(set_to_none=True)
            ep_loss += loss.item() if torch.isfinite(loss) else 0.0
            ep_prior += prior_val
        ep_loss /= max(len(trn_dl), 1)
        ep_prior /= max(len(trn_dl), 1)

        # ── val ──
        model.eval()
        vl, vd, vh, cnt = 0., 0., 0., 0
        with torch.no_grad():
            for batch in tqdm(val_dl, desc=f"E{epoch:03d} val", leave=False):
                m5 = batch["masks_5ch"].to(device); B = m5.size(0)
                with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
                    hull = compute_visual_hull(m5, device)
                vol = model(m5, hull)

                vol = vol.float()
                for i in range(5):
                    th = torch.full((B,), val_angles[i], device=device)
                    proj = render_projection(vol, th)
                    dl = dice_loss(proj, m5[:, i:i+1])
                    vl += dl.item(); vd += (1-dl).item()
                    pb = (proj>0.5).cpu().numpy()
                    mb = (m5[:, i:i+1]>0.5).cpu().numpy()
                    for b in range(B): vh += hd95(pb[b,0], mb[b,0]); cnt += 1
        vl /= max(len(val_dl)*5,1); vd /= max(len(val_dl)*5,1)
        vh /= max(cnt,1)
        elapsed = time.time()-t0

        hist["epoch"].append(epoch)
        hist["train_dice_loss"].append(ep_loss)
        hist["val_dice_loss"].append(vl)
        hist["val_dice_score"].append(vd)
        hist["val_hd95"].append(vh)
        hist["prior_loss"].append(ep_prior)
        sched.step(vd)

        lr_now = opt.param_groups[0]["lr"]
        print(f"E{epoch:03d} | loss={ep_loss:.4f} prior={ep_prior:.4f} | "
              f"vl={vl:.4f} vd={vd:.4f} hd={vh:.2f} | lr={lr_now:.6f} | {elapsed:.1f}s")

        # ── Checkpoint (new filenames for U-Net model) ──
        ckpt = {"epoch": epoch, "model": model.state_dict(),
                "opt": opt.state_dict(), "best_dice": max(best_dice,vd),
                "cfg": cfg, "hist": hist}
        torch.save(ckpt, Path(cfg["ckpt_dir"])/"3dbreastnet_unet_last.pth")
        if vd > best_dice:
            best_dice = vd; no_imp = 0
            torch.save(ckpt, Path(cfg["ckpt_dir"])/"3dbreastnet_unet_best.pth")
            print(f"  ★ new best dice={best_dice:.4f}")
        else:
            no_imp += 1
            if no_imp >= cfg["patience"]:
                print(f"Early stopping @ epoch {epoch}"); break

    # ── plot ──
    fig, ax = plt.subplots(1, 4, figsize=(24, 5))
    ax[0].plot(hist["epoch"], hist["train_dice_loss"], label="train")
    ax[0].plot(hist["epoch"], hist["val_dice_loss"], label="val")
    ax[0].set_title("Dice Loss"); ax[0].legend()
    ax[1].plot(hist["epoch"], hist["val_dice_score"], color="green")
    ax[1].set_title("Val Dice Score")
    ax[2].plot(hist["epoch"], hist["val_hd95"], color="red")
    ax[2].set_title("Val HD95")
    ax[3].plot(hist["epoch"], hist["prior_loss"], color="purple")
    ax[3].set_title("Shape Prior Loss")
    for a in ax: a.set_xlabel("Epoch")
    plt.tight_layout()
    plt.savefig(Path(cfg["ckpt_dir"])/"training_history.png", dpi=150)
    print(f"Plot saved. Best dice={best_dice:.4f}")

if __name__ == "__main__":
    train(CFG)

## View Training Metrics (Local Laptop)
Run this cell to extract the loss and dice curves from the saved checkpoint.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# ## CELL G — Evaluation Metrics (NEW — after training/plotting cells)
# ════════════════════════════════════════════════════════════════════════════

import numpy as np
from scipy.spatial.distance import directed_hausdorff

def compute_metrics(pred_proj, target_sil, threshold=0.5):
    """
    pred_proj:   (H, W) float in [0, 1]  — rendered 2D projection
    target_sil:  (H, W) binary           — input silhouette
    Returns dict with Accuracy, Dice, Jaccard, Hausdorff.
    """
    pred_bin = (pred_proj >= threshold).astype(np.float32)
    tgt      = target_sil.astype(np.float32)

    TP = (pred_bin * tgt).sum()
    TN = ((1 - pred_bin) * (1 - tgt)).sum()
    FP = (pred_bin * (1 - tgt)).sum()
    FN = ((1 - pred_bin) * tgt).sum()
    N  = pred_bin.size

    accuracy = (TP + TN) / N
    dice     = (2 * TP) / (2 * TP + FP + FN + 1e-6)
    jaccard  = TP / (TP + FP + FN + 1e-6)

    pred_pts = np.argwhere(pred_bin > 0).astype(float)
    tgt_pts  = np.argwhere(tgt  > 0).astype(float)
    if len(pred_pts) == 0 or len(tgt_pts) == 0:
        hausdorff = float('nan')
    else:
        hausdorff = max(
            directed_hausdorff(pred_pts, tgt_pts)[0],
            directed_hausdorff(tgt_pts,  pred_pts)[0]
        )

    return {
        'Accuracy':           accuracy,
        'Dice Index':         dice,
        'Jaccard Index':      jaccard,
        'Hausdorff Distance': hausdorff,
    }


# ── Evaluation loop ──
model.eval()
view_names   = ['RL (−90°)', 'RO (−45°)', 'F (0°)', 'LO (+45°)', 'LL (+90°)']
std_angles   = [-90, -45, 0, 45, 90]
results_per_view = {v: [] for v in view_names}

with torch.no_grad():
    for batch in val_dl:          # uses val_dl from training; swap to test_loader if available
        masks = batch["masks_5ch"].to(device)     # (B, 5, H, W)
        hull  = compute_visual_hull(masks, device)
        V_pred = model(masks, hull)               # (B, 1, 128, 128, 128)

        for view_idx, (vname, angle) in enumerate(zip(view_names, std_angles)):
            proj = render_projection(V_pred, angle)
            proj_np = proj.squeeze(1).cpu().float().numpy()  # (B, H, W)
            sil_np  = masks[:, view_idx].cpu().numpy()       # (B, H, W)

            for b in range(proj_np.shape[0]):
                m = compute_metrics(proj_np[b], sil_np[b])
                results_per_view[vname].append(m)

# ── Print table ──
metric_keys = ['Accuracy', 'Dice Index', 'Jaccard Index', 'Hausdorff Distance']
print(f"{'View':<15} {'Accuracy':>10} {'Dice':>10} {'Jaccard':>10} {'Hausdorff':>12}")
print("-" * 60)
all_vals = {k: [] for k in metric_keys}

for vname in view_names:
    vals = results_per_view[vname]
    row  = {k: np.nanmean([v[k] for v in vals]) for k in metric_keys}
    for k in metric_keys:
        all_vals[k].extend([v[k] for v in vals])
    print(f"{vname:<15} {row['Accuracy']:>10.4f} {row['Dice Index']:>10.4f} "
          f"{row['Jaccard Index']:>10.4f} {row['Hausdorff Distance']:>12.4f}")

print("-" * 60)
overall = {k: np.nanmean(all_vals[k]) for k in metric_keys}
print(f"{'Overall':<15} {overall['Accuracy']:>10.4f} {overall['Dice Index']:>10.4f} "
      f"{overall['Jaccard Index']:>10.4f} {overall['Hausdorff Distance']:>12.4f}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# ## CELL H — Thermal Projection (NEW — after evaluation metrics)
# ════════════════════════════════════════════════════════════════════════════

import torch
import numpy as np
from scipy.spatial import cKDTree

def estimate_view_angle(V_pred_np, sil_np, angle_range=(-90, 90), step=1):
    """Find angle θ̂ minimising Dice loss between silhouette and projection."""
    best_angle, best_loss = 0, float('inf')
    angles = np.arange(angle_range[0], angle_range[1] + step, step)
    V_t = torch.from_numpy(V_pred_np).unsqueeze(0).unsqueeze(0).float()
    for angle in angles:
        proj = render_projection(V_t, angle).squeeze().numpy()
        inter = (proj * sil_np).sum()
        loss  = 1 - (2 * inter) / (proj.sum() + sil_np.sum() + 1e-6)
        if loss < best_loss:
            best_loss, best_angle = loss, angle
    return best_angle


def overlay_temperatures_on_volume(V_pred_np, thermal_images, estimated_angles,
                                   volume_size=128):
    """Maps 2D temperatures onto 3D silhouette surface via ray-casting."""
    D, H, W   = V_pred_np.shape
    temp_vol  = np.full((D, H, W), np.nan, dtype=np.float32)
    count_vol = np.zeros((D, H, W), dtype=np.float32)
    vox_coords = np.argwhere(V_pred_np > 0.5).astype(np.float32)

    for angle, temp_2d in zip(estimated_angles, thermal_images):
        if temp_2d is None:
            continue
        theta = np.radians(angle)
        cos_t, sin_t = np.cos(theta), np.sin(theta)
        centre = np.array([D / 2, H / 2, W / 2])
        shifted = vox_coords - centre
        d, h, w = shifted[:, 0], shifted[:, 1], shifted[:, 2]
        w_rot =  w * cos_t + d * sin_t
        d_rot = -w * sin_t + d * cos_t
        front_mask = d_rot > 0
        if front_mask.sum() == 0:
            continue
        h_pix = (h[front_mask] + centre[1]).astype(int)
        w_pix = (w_rot[front_mask] + centre[2]).astype(int)
        temp_h, temp_w = temp_2d.shape
        scale_h, scale_w = temp_h / H, temp_w / W
        h_img = np.clip((h_pix * scale_h).astype(int), 0, temp_h - 1)
        w_img = np.clip((w_pix * scale_w).astype(int), 0, temp_w - 1)
        orig_coords = np.argwhere(V_pred_np > 0.5)[front_mask]
        for i, (vd, vh, vw) in enumerate(orig_coords):
            t = temp_2d[h_img[i], w_img[i]]
            if not np.isnan(t):
                if np.isnan(temp_vol[vd, vh, vw]):
                    temp_vol[vd, vh, vw]  = t
                    count_vol[vd, vh, vw] = 1
                else:
                    temp_vol[vd, vh, vw] = (
                        temp_vol[vd, vh, vw] * count_vol[vd, vh, vw] + t)
                    count_vol[vd, vh, vw] += 1
                    temp_vol[vd, vh, vw] /= count_vol[vd, vh, vw]

    # KNN interpolation for occluded voxels
    occupied_mask   = ~np.isnan(temp_vol) & (V_pred_np > 0.5)
    unoccupied_mask =  np.isnan(temp_vol) & (V_pred_np > 0.5)
    if occupied_mask.sum() > 0 and unoccupied_mask.sum() > 0:
        filled_coords = np.argwhere(occupied_mask).astype(np.float32)
        filled_temps  = temp_vol[occupied_mask]
        query_coords  = np.argwhere(unoccupied_mask).astype(np.float32)
        tree  = cKDTree(filled_coords)
        _, idx = tree.query(query_coords, k=1)
        temp_vol[unoccupied_mask] = filled_temps[idx]

    return temp_vol

## Post-Training 3D Visualization (Local Laptop)
Run this cell to perform inference using explicit local Windows paths.

In [ ]:
!uv pip install --upgrade nbformat

In [ ]:
# ════════════════════════════════════════════════════════════════
# 3D VISUALIZATION OF RANDOM PATIENTS (LAPTOP LOCAL)
# ════════════════════════════════════════════════════════════════
import random
import io
import numpy as np
from IPython.display import display, Image
import torch
import torch.nn.functional as F
from pathlib import Path
from skimage.measure import marching_cubes
from scipy.ndimage import gaussian_filter
import plotly.graph_objects as go
# Ensure models and dataset imports are available
from models_v6 import UNet, Encoder2D, Decoder3D
from train import build_patient_groups, PatientDataset
# Fallback if the model definition cell was not executed
try:
    compute_visual_hull
except NameError:
    import math
    def compute_visual_hull(m5, device):
        """Computes a 3D visual hull intersection from the 5 view masks."""
        B = m5.shape[0]
        D = H = W = 128
        z, y, x = torch.meshgrid(
            torch.linspace(-1, 1, D, device=device),
            torch.linspace(-1, 1, H, device=device),
            torch.linspace(-1, 1, W, device=device),
            indexing="ij",
        )
        grid_pts = torch.stack([x, y, z], dim=0).view(3, -1)
        angles = [-90.0, -45.0, 0.0, 45.0, 90.0]
        hull = torch.ones((B, 1, D * H * W), device=device)
        for i, angle in enumerate(angles):
            rad = angle * math.pi / 180.0
            c, s = math.cos(rad), math.sin(rad)
            x_cam = grid_pts[0] * c - grid_pts[2] * s
            y_cam = grid_pts[1]
            sample_coords = torch.stack([x_cam, y_cam], dim=-1).unsqueeze(0).unsqueeze(2).expand(B, -1, -1, -1)
            mask_view = m5[:, i:i + 1, :, :]
            sampled = F.grid_sample(mask_view, sample_coords, mode="bilinear", padding_mode="zeros", align_corners=True)
            hull = hull * sampled.squeeze(3)
        return hull.view(B, 1, D, H, W)
def visualize_random_patients_local(n=5):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    # Hardcoded Absolute Paths for your Laptop
    unet_ckpt = r"/mnt/Data1/Peoples/faiz836b/DNP-3DDMR-IR/UNET_Segmentation/breast_segmentation_unet_best_gpu.pth"
    tiff_base = r"/mnt/Data1/Peoples/faiz836b/DNP-3DDMR-IR/data/organized_by_patient"
    ckpt_path = Path(r"/mnt/Data1/Peoples/faiz836b/DNP-3DDMR-IR/UNET_Segmentation/3DBreastnet/checkpoints_3d_v5/3dbreastnet_best.pth")
    
    if not ckpt_path.exists():
        print(f"Error: No trained model found at {ckpt_path}.")
        return
        
    print("Loading checkpoints...")
    
    unet = UNet().to(device)
    unet.load_state_dict(torch.load(unet_ckpt, map_location=device, weights_only=False))
    unet.eval()
    
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    enc = Encoder2D().to(device)
    dec = Decoder3D().to(device)
    enc.load_state_dict(ckpt["enc"])
    dec.load_state_dict(ckpt["dec"], strict=False)
    enc.eval(); dec.eval()
    
    print(f"Loading patient list from {tiff_base}...")
    groups = build_patient_groups(tiff_base)
    if len(groups) == 0: 
        print("No patients found. Check dataset path.")
        return
    
    selected = random.sample(groups, min(n, len(groups)))
    dataset = PatientDataset(selected, unet, device)
    
    print(f"Generating 3D models for {len(selected)} patients...")
    
    for i in range(len(dataset)):
        item = dataset[i]
        pid, label = item["patient_id"], item["label"]
        m5 = item["masks_5ch"].unsqueeze(0).to(device)
        
        with torch.no_grad(), torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            hull = compute_visual_hull(m5, device)
            try:
                vol = dec(enc(m5), hull)
            except TypeError:
                vol = dec(enc(m5))
        vol_np = vol[0, 0].float().cpu().numpy()
        vmin, vmax, vmean = float(vol_np.min()), float(vol_np.max()), float(vol_np.mean())
        print(f"{pid}: min={vmin:.4f} max={vmax:.4f} mean={vmean:.4f}")
        if vmax <= 1e-6:
            print(f"Could not generate 3D mesh for {pid} (volume is empty).")
            continue
        vol_np = gaussian_filter(vol_np, sigma=1.5)
        
        # Tighter surface with extra smoothing
        levels = [
            0.12 * vmax,
            0.18 * vmax,
            0.25 * vmax,
            float(np.percentile(vol_np, 80)),
            float(np.percentile(vol_np, 90)),
            float(np.percentile(vol_np, 95)),
            float(np.percentile(vol_np, 98)),
            0.6 * vmax,
        ]
        levels = [lv for lv in levels if vmin < lv < vmax]
        if not levels:
            print(f"Could not generate 3D mesh for {pid} (no valid level between min/max).")
            continue
        
        min_faces = 2000
        verts = faces = None
        used_level = None
        best_faces = -1
        for level in levels:
            try:
                verts_try, faces_try, normals, values = marching_cubes(vol_np, level=level)
                face_count = faces_try.shape[0]
                if face_count > best_faces:
                    verts, faces = verts_try, faces_try
                    used_level = level
                    best_faces = face_count
                if face_count >= min_faces:
                    break
            except ValueError:
                continue
        
        if verts is None:
            print(f"Could not generate 3D mesh for {pid} (tried {len(levels)} levels).")
            continue
        
        fig = go.Figure(data=[
            go.Mesh3d(
                x=verts[:, 0], y=verts[:, 1], z=verts[:, 2],
                i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
                colorscale='Hot',
                intensity=verts[:, 2],
                showscale=False
            )
        ])
        fig.update_layout(
            title=f"Patient: {pid} | Label: {label} (3D Reconstruction, level={used_level:.3f}, faces={faces.shape[0]})",
            scene=dict(
                xaxis_title='X', yaxis_title='Y', zaxis_title='Z',
                aspectmode='data'
            ),
            margin=dict(l=0, r=0, b=0, t=40)
        )
        fig.show()
# Run it!
visualize_random_patients_local(n=5)


## Export Geometry (.STL)
Run this cell to batch process **all available patients** through the network and export their 3D reconstructed geometries into universally readable `.stl` files. You can open these in Blender, MeshLab, or Windows 3D Viewer.

In [ ]:
# ════════════════════════════════════════════════════════════════
# BATCH EXPORT ALL RECONSTRUCTED GEOMETRIES TO .STL
# ════════════════════════════════════════════════════════════════
import os
import struct
import torch
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm
from scipy.ndimage import binary_closing, binary_fill_holes, generate_binary_structure, label, gaussian_filter
from skimage.measure import marching_cubes
# Ensure models and dataset imports are available
from models_v6 import UNet, Encoder2D, Decoder3D
from train import build_patient_groups, PatientDataset
def clean_volume_for_export(volume, threshold=0.35):
    """Convert the raw prediction into a closed binary volume before meshing."""
    mask = volume >= threshold
    structure = generate_binary_structure(3, 2)
    # Fill gaps and seal small openings so the exported mesh is watertight.
    mask = binary_closing(mask, structure=structure, iterations=1)
    mask = binary_fill_holes(mask)
    labeled, num = label(mask, structure=structure)
    if num > 1:
        sizes = np.bincount(labeled.ravel())
        sizes[0] = 0
        mask = labeled == sizes.argmax()
    # Pad one voxel so marching_cubes can build a closed outer shell.
    mask = np.pad(mask, 1, mode="constant", constant_values=False)
    return mask.astype(np.float32)
def save_stl(filename, verts, faces):
    """Writes a binary STL file natively without needing external libraries."""
    with open(filename, "wb") as f:
        # 80 byte header
        f.write(b"\0" * 80)
        # Number of triangles (uint32)
        f.write(struct.pack("<I", len(faces)))
        for face in faces:
            tri = verts[face].astype(np.float32, copy=False)
            v0, v1, v2 = tri
            normal = np.cross(v1 - v0, v2 - v0)
            norm = np.linalg.norm(normal)
            if norm > 0:
                normal = (normal / norm).astype(np.float32, copy=False)
            else:
                normal = np.zeros(3, dtype=np.float32)
            f.write(struct.pack("<3f", *normal))
            f.write(struct.pack("<3f", *v0))
            f.write(struct.pack("<3f", *v1))
            f.write(struct.pack("<3f", *v2))
            # Attribute byte count (uint16)
            f.write(struct.pack("<H", 0))
def export_all_patients_to_stl():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # Setup Output Directory
    out_dir = Path("exported_stls_v5")
    out_dir.mkdir(exist_ok=True)
    unet_ckpt = r"../../breast_segmentation_unet_best_gpu.pth"
    tiff_base = r"../../data/organized_by_patient"
    ckpt_path = Path(Path("checkpoints_3d_v5/3dbreastnet_best.pth"))
    if not ckpt_path.exists():
        print(f"Error: No trained model found at {ckpt_path}.")
        return
    print("Loading checkpoints...")
    unet = UNet().to(device)
    unet.load_state_dict(torch.load(unet_ckpt, map_location=device, weights_only=False))
    unet.eval()
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    enc = Encoder2D().to(device)
    dec = Decoder3D().to(device)
    enc.load_state_dict(ckpt["enc"])
    dec.load_state_dict(ckpt["dec"])
    enc.eval(); dec.eval()
    groups = build_patient_groups(tiff_base)
    if not groups:
        return
    dataset = PatientDataset(groups, unet, device)
    print(f"Batch exporting {len(dataset)} patients to .STL...")
    for i in tqdm(range(len(dataset))):
        item = dataset[i]
        pid = item["patient_id"]
        m5 = item["masks_5ch"].unsqueeze(0).to(device)
        with torch.no_grad(), torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            hull = compute_visual_hull(m5, device)
            vol = dec(enc(m5), hull).float()
            
        vol_np = vol[0, 0].cpu().numpy()
        
        # 1. Pad the volume with 0s on all boundaries to force a closed, watertight solid!
        import numpy as np
        vol_np = np.pad(vol_np, pad_width=1, mode='constant', constant_values=0)
        
        # 2. Apply stronger Gaussian smoothing to eliminate the voxel staircasing (sigma=2.0)
        vol_np = gaussian_filter(vol_np, sigma=2.0)
        
        try:
            verts, faces, normals, _ = marching_cubes(vol_np, level=0.5)
            # 3. Shift vertices back by 1 to compensate for the padding
            verts = verts - 1.0
            
            # Scale coordinates relative to [-1, 1] for sane mesh sizes in external viewers
            verts = (verts / 63.5) - 1.0
            
            # Export
            export_path = out_dir / f"{pid}_3d_geometry.stl"
            save_stl(export_path, verts, faces, normals)
        except Exception as e:
            print(f"Skipping {pid} due to error: {e}")
    print(f"\nSuccess! All meshes have been saved to the '{out_dir.absolute()}' directory.")
# Run the batch exporter
export_all_patients_to_stl()


## Test Single Patient STL Export
Run this cell to quickly generate and test the `.stl` output for a specific patient without batch processing the whole dataset.

In [ ]:
# ════════════════════════════════════════════════════════════════
# BATCH EXPORT ALL RECONSTRUCTED GEOMETRIES TO .STL
# ════════════════════════════════════════════════════════════════
import os
import torch
import struct
from pathlib import Path
import numpy as np
from scipy.ndimage import gaussian_filter
from skimage.measure import marching_cubes
from models_v6 import UNet, Encoder2D, Decoder3D
from train import build_patient_groups, PatientDataset
def save_stl(filename, verts, faces, normals):
    with open(filename, "wb") as f:
        f.write(b"\0" * 80)
        f.write(struct.pack("<I", len(faces)))
        for face in faces:
            f.write(struct.pack("<3f", 0.0, 0.0, 0.0))
            f.write(struct.pack("<3f", *verts[face[0]]))
            f.write(struct.pack("<3f", *verts[face[1]]))
            f.write(struct.pack("<3f", *verts[face[2]]))
            f.write(struct.pack("<H", 0))
def export_all_patients_to_stl():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    out_dir = Path("exported_stls_v5")
    out_dir.mkdir(exist_ok=True)
    
    unet_ckpt = r"../../breast_segmentation_unet_best_gpu.pth"
    tiff_base = r"../../data/organized_by_patient"
    ckpt_path = Path(Path("checkpoints_3d_v5/3dbreastnet_best.pth"))
    
    if not ckpt_path.exists():
        print(f"Error: Checkpoint not found at {ckpt_path}.")
        return
        
    print("Loading checkpoints...")
    unet = UNet().to(device)
    unet.load_state_dict(torch.load(unet_ckpt, map_location=device, weights_only=False))
    unet.eval()
    
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    enc = Encoder2D().to(device)
    dec = Decoder3D().to(device)
    enc.load_state_dict(ckpt["enc"])
    dec.load_state_dict(ckpt["dec"])
    enc.eval(); dec.eval()
    
    groups = build_patient_groups(tiff_base)
    if not groups:
        print("No complete patients found in the dataset.")
        return
        
    dataset = PatientDataset(groups, unet, device)
    print(f"Batch exporting {len(dataset)} patients to .STL...")
    
    for item in dataset:
        pid = item["patient_id"]
        m5 = item["masks_5ch"].unsqueeze(0).to(device)
        
        with torch.no_grad(), torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            hull = compute_visual_hull(m5, device)
            vol = dec(enc(m5), hull).float()
            
        vol_np = vol[0, 0].cpu().numpy()
        vol_np = np.pad(vol_np, pad_width=1, mode='constant', constant_values=0)
        vol_np = gaussian_filter(vol_np, sigma=2.0)
        
        try:
            verts, faces, normals, _ = marching_cubes(vol_np, level=0.5)
            verts = verts - 1.0
            verts = (verts / 63.5) - 1.0
            
            export_path = out_dir / f"{pid}_3d_geometry.stl"
            save_stl(export_path, verts, faces, normals)
            print(f"Exported {pid} -> {export_path}")
        except Exception as e:
            print(f"Skipping {pid} due to error: {e}")
    print(f"\nSuccess! All meshes have been saved to the '{out_dir.absolute()}' directory.")
# Run the batch exporter
export_all_patients_to_stl()


## Per Patient Projection Validation on Each View

In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm
# Import model architectures and data loaders from your existing scripts
from models_v6 import UNet, Encoder2D, Decoder3D
from train import build_patient_groups, PatientDataset, dice_loss, render_projection
def evaluate_all_patients():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    # 1. Setup Directories
    out_dir = Path("projection_plots_v5")
    out_dir.mkdir(exist_ok=True)
    
    # 2. Paths
    base_dir = Path("../..").resolve()
    unet_ckpt = base_dir / "UNET_Segmentation" / "breast_segmentation_unet_best_gpu.pth"
    tiff_base = base_dir / "data" / "organized_by_patient"
    ckpt_path = Path("checkpoints_3d_v5/3dbreastnet_best.pth")
    
    # 3. Load Models
    print("Loading checkpoints...")
    unet = UNet().to(device)
    unet.load_state_dict(torch.load(unet_ckpt, map_location=device, weights_only=False))
    unet.eval()
    
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    enc = Encoder2D().to(device)
    dec = Decoder3D().to(device)
    enc.load_state_dict(ckpt["enc"])
    dec.load_state_dict(ckpt["dec"])
    enc.eval(); dec.eval()
    
    # 4. Load Data
    groups = build_patient_groups(tiff_base)
    if not groups:
        print("No patients found.")
        return
    dataset = PatientDataset(groups, unet, device)
    
    # 5. Evaluation Loop
    view_angles = [-90.0, -45.0, 0.0, 45.0, 90.0]
    view_names = ["Right Lateral (-90°)", "Right Oblique (-45°)", "Frontal (0°)", "Left Oblique (45°)", "Left Lateral (90°)"]
    
    all_dices = []
    
    print(f"\nEvaluating {len(dataset)} patients and generating 2x5 plots...")
    
    for item in tqdm(dataset):
        pid = item["patient_id"]
        m5 = item["masks_5ch"].unsqueeze(0).to(device)  # Ground Truth 2D Silhouettes [1, 5, 128, 128]
        
        with torch.no_grad(), torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            # Generate 3D Volume
            hull = compute_visual_hull(m5, device)
            vol = dec(enc(m5), hull).float()
            
            patient_dices = []
            projections = []
            
            # Render and Calculate Dice for the 5 standard angles
            for i, angle in enumerate(view_angles):
                th = torch.tensor([angle], device=device, dtype=torch.float32)
                proj = render_projection(vol, th)  # [1, 1, 128, 128]
                
                gt_mask = m5[:, i:i+1] # [1, 1, 128, 128]
                dl = dice_loss(proj, gt_mask)
                d_score = 1.0 - dl.item()
                
                patient_dices.append(d_score)
                # Binarize the projection at 0.5 threshold to see the sharp silhouette curves
                binary_proj = (proj[0, 0] > 0.5).cpu().numpy().astype(np.float32)
                projections.append(binary_proj)
        
        all_dices.append(np.mean(patient_dices))
        
        # 6. Generate 2x5 Grid Plot
        fig, axes = plt.subplots(2, 5, figsize=(20, 8))
        fig.suptitle(f"Patient: {pid} | Average Dice Score: {np.mean(patient_dices):.4f}", fontsize=16)
        
        for i in range(5):
            # Top Row: Ground Truth (U-Net mask)
            gt_img = m5[0, i].cpu().numpy()
            axes[0, i].imshow(gt_img, cmap='gray')
            axes[0, i].set_title(f"GT: {view_names[i]}")
            axes[0, i].axis('off')
            
            # Bottom Row: 3D Projected Silhouette
            axes[1, i].imshow(projections[i], cmap='gray')
            axes[1, i].set_title(f"Predicted Projection (Dice: {patient_dices[i]:.3f})")
            axes[1, i].axis('off')
            
        plt.tight_layout()
        plot_path = out_dir / f"{pid}_validation_grid.png"
        plt.savefig(plot_path, dpi=150)
        plt.close(fig)
        
    # 7. Final Results
    overall_mean = np.mean(all_dices)
    overall_std = np.std(all_dices)
    
    print("\n" + "="*50)
    print("FINAL VALIDATION RESULTS (Test on All Patients)")
    print("="*50)
    print(f"Total Patients Evaluated : {len(dataset)}")
    print(f"Overall Mean Dice Score  : {overall_mean:.4f}")
    print(f"Standard Deviation       : {overall_std:.4f}")
    print(f"Validation plots saved in: {out_dir.absolute()}")
if __name__ == "__main__":
    evaluate_all_patients()
